# Image Classification & Transfer Learning Pipelines

This interactive lab covers:
1. **ResNet Architecture**: Residual connections, feature extraction, and global average pooling.
2. **Feature Embeddings**: Extracting dense representations before the classification head.
3. **Transfer Learning Dynamics**: Linear probing (frozen backbone) vs. end-to-end fine-tuning.
4. **Evaluation Metrics**: Multi-class Top-1 and Top-5 accuracy computation.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('04-computer-vision/image-classification/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
import torch.optim as optim
from classifier import (
    ImageClassifier,
    TransferLearningWrapper,
    compute_accuracy_topk,
)

print('Image Classification & Transfer Learning module loaded successfully!')

## 1. Backbone Architecture & Feature Representation

We instantiate a lightweight residual network and inspect both the dense pooled feature vectors and class logits.

In [ ]:
torch.manual_seed(42)
model = ImageClassifier(in_channels=3, num_classes=10, base_dim=16)

# Synthetic batch of 4 images (C=3, H=32, W=32)
x = torch.randn(4, 3, 32, 32)
targets = torch.tensor([2, 5, 0, 7])

out = model(x, targets=targets)
print(f'Input image batch shape:        {x.shape}')
print(f'Dense feature embedding shape:  {out["features"].shape}')
print(f'Output class logits shape:      {out["logits"].shape}')
print(f'Initial Cross-Entropy Loss:     {out["loss"].item():.4f}')

## 2. Transfer Learning: Linear Probing vs. Fine-Tuning

In Linear Probing, we freeze the backbone weights and train solely the new classification head for a 3-class target domain.

In [ ]:
tl_model = TransferLearningWrapper(model, target_classes=3)

# Step 1: Linear Probe mode (Backbone Frozen)
tl_model.freeze_backbone()
trainable_params = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in tl_model.parameters())
print(f'Linear Probing - Trainable parameters: {trainable_params:,} / {total_params:,}')

# Optimize single batch on new 3-class target task
target_y = torch.tensor([0, 1, 2, 1])
optimizer = optim.Adam(tl_model.new_head.parameters(), lr=1e-2)
for step in range(5):
    optimizer.zero_grad()
    step_out = tl_model(x, targets=target_y)
    step_out['loss'].backward()
    optimizer.step()
print(f'Linear Probe Loss after 5 steps: {step_out["loss"].item():.4f}')

# Step 2: Unfreeze for Fine-Tuning
tl_model.unfreeze_backbone()
trainable_ft = sum(p.numel() for p in tl_model.parameters() if p.requires_grad)
print(f'Full Fine-Tuning - Trainable parameters: {trainable_ft:,} / {total_params:,}')

## 3. Evaluation Metrics: Top-1 and Top-k Accuracy

Computing Top-1 and Top-2 accuracy across model predictions.

In [ ]:
tl_model.eval()
with torch.no_grad():
    eval_out = tl_model(x)
    top1, top2 = compute_accuracy_topk(eval_out['logits'], target_y, topk=(1, 2))

print(f'Top-1 Accuracy: {top1:.1f}%')
print(f'Top-2 Accuracy: {top2:.1f}%')